In [1]:
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec
import datetime
from reportlab.lib.pagesizes import A4, landscape
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.lib.utils import ImageReader
from reportlab.lib import colors as rl_colors
import io
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
OUTDIR = r"D:\report"
os.makedirs(OUTDIR, exist_ok=True)

epsilon = 1e-6
# ─────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────

def load_band(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Image not found: {path}")
    if len(img.shape) == 3:
        img = img[:, :, 0]
    return img.astype(np.float32)


def normalize(img):
    img = img.astype(np.float32)
    return (img - np.min(img)) / (np.max(img) - np.min(img) + epsilon)


def norm_full(arr, low=5, high=95):
    vals = arr.flatten()
    p_low  = np.percentile(vals, low)
    p_high = np.percentile(vals, high)
    out = (arr - p_low) / (p_high - p_low + epsilon)
    return np.clip(out, 0, 1)


def make_overlay(norm_map, cmap, alpha=0.7):
    colored = cmap(norm_map)
    colored[:, :, 3] = alpha
    return colored


def fig_to_img(fig):
    """Convert a matplotlib figure to a numpy RGB array."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0)
    arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    buf.close()
    return img_rgb

In [3]:
 
# ─────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────

def load_band(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"❌ Image not found: {path}")
    if len(img.shape) == 3:
        img = img[:, :, 0]
    return img.astype(np.float32)


def normalize(img):
    return (img - np.min(img)) / (np.max(img) - np.min(img) + epsilon)


def norm_full(arr, low=5, high=95):
    vals = arr.flatten()
    p_low  = np.percentile(vals, low)
    p_high = np.percentile(vals, high)
    out = (arr - p_low) / (p_high - p_low + epsilon)
    return np.clip(out, 0, 1)


def make_overlay(norm_map, cmap, alpha=0.7):
    colored = cmap(norm_map)
    colored[:, :, 3] = alpha
    return colored


def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0)
    arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    buf.close()
    return img_rgb

In [ ]:
# ─────────────────────────────────────────────
# Band loader (CLEAN + SCALABLE)
# ─────────────────────────────────────────────
DIR = r"D:\HSI_OUTPUT"

def get_band(n):
    path = os.path.join(DIR, f"Band_{n:02d}.png")
    return load_band(path)


# ─────────────────────────────────────────────
# Load required bands
# ─────────────────────────────────────────────
blue    = get_band(8)
green   = get_band(17)
red     = get_band(27)
rededge = get_band(30)
nir     = get_band(31)


# ─────────────────────────────────────────────
# OPTIONAL: Load ALL 31 bands (for ML / FPGA)
# ─────────────────────────────────────────────
bands = [get_band(i) for i in range(1, 32)]


# ─────────────────────────────────────────────
# Example: Create RGB image
# ─────────────────────────────────────────────
rgb = np.stack([
    normalize(red),
    normalize(green),
    normalize(blue)
], axis=-1)


# ─────────────────────────────────────────────
# Example: NDVI (important for your project)
# ─────────────────────────────────────────────
ndvi = (nir - red) / (nir + red + epsilon)
ndvi_norm = norm_full(ndvi)


# ─────────────────────────────────────────────
# Visualization
# ─────────────────────────────────────────────
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.title("RGB Image")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1,2,2)
plt.title("NDVI")
plt.imshow(ndvi_norm, cmap="jet")
plt.colorbar()

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# Save outputs
# ─────────────────────────────────────────────
cv2.imwrite(os.path.join(OUTDIR, "rgb.png"), (rgb * 255).astype(np.uint8))
cv2.imwrite(os.path.join(OUTDIR, "ndvi.png"), (ndvi_norm * 255).astype(np.uint8))


# ─────────────────────────────────────────────
# Debug (VERY IMPORTANT)
# ─────────────────────────────────────────────
print("Bands loaded successfully")
print("RGB shape:", rgb.shape)
#print("NDVI min/max:", ndvi.min(), ndvi.max())

# ─────────────────────────────────────────────
# Resize to common shape
# ─────────────────────────────────────────────

h = min(blue.shape[0], green.shape[0], red.shape[0], rededge.shape[0], nir.shape[0])
w = min(blue.shape[1], green.shape[1], red.shape[1], rededge.shape[1], nir.shape[1])

blue    = blue[:h, :w]
green   = green[:h, :w]
red     = red[:h, :w]
rededge = rededge[:h, :w]
nir     = nir[:h, :w]

# ─────────────────────────────────────────────
# Normalize reflectance
# ─────────────────────────────────────────────

blue    = normalize(blue)
green   = normalize(green)
red     = normalize(red)
rededge = normalize(rededge)
nir     = normalize(nir)

# ─────────────────────────────────────────────
# Foot segmentation (remove background)
# ─────────────────────────────────────────────

mask   = (nir > 0.25).astype(np.float32)
kernel = np.ones((7, 7), np.uint8)
mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
mask   = cv2.GaussianBlur(mask, (9, 9), 0)

blue    *= mask
green   *= mask
red     *= mask
rededge *= mask
nir     *= mask

# ─────────────────────────────────────────────
# Clip reflectance
# ─────────────────────────────────────────────

min_ref = 0.01
blue    = np.clip(blue,    min_ref, 1)
green   = np.clip(green,   min_ref, 1)
red     = np.clip(red,     min_ref, 1)
rededge = np.clip(rededge, min_ref, 1)
nir     = np.clip(nir,     min_ref, 1)

# ─────────────────────────────────────────────
# Beer-Lambert absorption
# ─────────────────────────────────────────────

abs_blue    = np.log(1 / blue)
abs_green   = np.log(1 / green)
abs_red     = np.log(1 / red)
abs_rededge = np.log(1 / rededge)
abs_nir     = np.log(1 / nir)

# ─────────────────────────────────────────────
# Biomarker computation
# ─────────────────────────────────────────────

HbO2      = 0.65 * abs_nir + 0.25 * abs_rededge + 0.10 * abs_green
Hb        = 0.70 * abs_red  + 0.30 * abs_green
Melanin   = abs_blue
StO2      = HbO2 / (HbO2 + Hb + epsilon)
Perfusion = abs_green
Stress    = abs_rededge

# ─────────────────────────────────────────────
# Smooth biomarkers
# ─────────────────────────────────────────────

HbO2      = cv2.GaussianBlur(HbO2,      (7, 7), 0)
Hb        = cv2.GaussianBlur(Hb,        (7, 7), 0)
StO2      = cv2.GaussianBlur(StO2,      (7, 7), 0)
Perfusion = cv2.GaussianBlur(Perfusion, (7, 7), 0)
Stress    = cv2.GaussianBlur(Stress,    (7, 7), 0)

# ─────────────────────────────────────────────
# Ulcer risk model
# ─────────────────────────────────────────────

Risk = (
      0.45 * (1 - norm_full(StO2))
    + 0.35 * (1 - norm_full(Perfusion))
    + 0.20 *      norm_full(Stress)
)

# ─────────────────────────────────────────────
# Normalize for visualization
# ─────────────────────────────────────────────

HbO2_n    = norm_full(HbO2)
Hb_n      = norm_full(Hb)
Melanin_n = norm_full(Melanin)
StO2_n    = norm_full(StO2)
Perf_n    = norm_full(Perfusion)
Stress_n  = norm_full(Stress)
Risk_n    = norm_full(Risk)

# ─────────────────────────────────────────────
# Clinical statistics
# ─────────────────────────────────────────────

toe_zone  = StO2[:h // 3, :]
mid_zone  = StO2[h // 3:2 * h // 3, :]
heel_zone = StO2[2 * h // 3:, :]

toe_risk  = Risk_n[:h // 3, :]
mid_risk  = Risk_n[h // 3:2 * h // 3, :]
heel_risk = Risk_n[2 * h // 3:, :]

mean_StO2 = float(np.mean(StO2))
mean_perf = float(np.mean(Perf_n))
mean_risk = float(np.mean(Risk_n))
max_risk  = float(np.max(Risk_n))

toe_sto2_mean  = float(toe_zone.mean())
mid_sto2_mean  = float(mid_zone.mean())
heel_sto2_mean = float(heel_zone.mean())

toe_risk_mean  = float(toe_risk.mean())
mid_risk_mean  = float(mid_risk.mean())
heel_risk_mean = float(heel_risk.mean())

zone_imbal = max(toe_sto2_mean, mid_sto2_mean, heel_sto2_mean) - \
             min(toe_sto2_mean, mid_sto2_mean, heel_sto2_mean)

# ─────────────────────────────────────────────
# Assessment thresholds
# ─────────────────────────────────────────────

 
if mean_risk <= 0.50:
    assessment = "NORMAL"
    acolor_mpl = "yellow"
    acolor_hex = "#FFD700"

else:
    assessment = "HIGH ULCER RISK — CLINICAL ATTENTION NEEDED"
    acolor_mpl = "red"
    acolor_hex = "#FF0000"

# ─────────────────────────────────────────────
# RGB base image
# ─────────────────────────────────────────────

rgb = np.stack([red, green, blue], axis=-1)
rgb = np.clip(rgb, 0, 1)

# ═════════════════════════════════════════════
# FIGURE 1 — Full Spectral Biomarker Atlas
# ═════════════════════════════════════════════

DARK_BG = "#0d0d0d"

biomarkers = [
    (HbO2_n,    plt.cm.hot,       "HbO\u2082\n(Oxyhemoglobin)",       "low \u2190 HbO\u2082 \u2192 high"),
    (Hb_n,      plt.cm.Blues_r,   "Hb\n(Deoxyhemoglobin)",            "low \u2190 Hb \u2192 high"),
    (Melanin_n, plt.cm.copper,    "Melanin",                           "low \u2190 Melanin \u2192 high"),
    (StO2_n,    plt.cm.RdYlGn,    "StO\u2082\n(O\u2082 Saturation)",   "red=low  green=high"),
    (Perf_n,    plt.cm.cool,      "Perfusion\n(Blood Flow)",           "low \u2190 flow \u2192 high"),
    (Stress_n,  plt.cm.YlOrRd,    "Tissue Stress\n(Red-edge)",         "yellow=low  red=high"),
    (Risk_n,    plt.cm.RdYlGn_r,  "Ulcer Risk\nScore",                 "green=safe  red=risk"),
]

N = len(biomarkers)       # 7 columns (+ 1 RGB)
TOTAL_COLS = N + 1        # 8 columns

# We show: row-0 = overlay, row-1 = raw map, row-2 = overlay2 (same foot, 3rd crop)
# Colorbar row at bottom
fig1 = plt.figure(figsize=(28, 11), facecolor=DARK_BG)

# Grid: 3 image rows + 1 colorbar row
ROW_HEIGHTS = [4, 4, 4, 0.55]
gs = GridSpec(4, TOTAL_COLS, figure=fig1,
              hspace=0.06, wspace=0.04,
              height_ratios=ROW_HEIGHTS,
              top=0.93, bottom=0.04, left=0.01, right=0.99)


def _dark_ax(ax):
    ax.set_facecolor(DARK_BG)
    ax.axis("off")


def _plot_col(row, col, data, cmap, overlay=True, title=None, label_size=7):
    ax = fig1.add_subplot(gs[row, col])
    _dark_ax(ax)
    if overlay:
        ax.imshow(rgb)
        ax.imshow(make_overlay(data, cmap, alpha=0.75))
    else:
        ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
    if title:
        ax.set_title(title, color="white", fontsize=label_size,
                     fontfamily="monospace", pad=3)
    return ax


# ── Column 0 = RGB (all three rows)
for row in range(3):
    ax = fig1.add_subplot(gs[row, 0])
    _dark_ax(ax)
    ax.imshow(rgb)
    row_label = "Original\nRGB"
    ax.set_title(row_label, color="white", fontsize=7,
                 fontfamily="monospace", pad=3)

# ── Columns 1..7 = biomarkers
for ci, (data, cmap, title, _) in enumerate(biomarkers):
    col = ci + 1
    # Row 0: overlay
    _plot_col(0, col, data, cmap, overlay=True,  title=title, label_size=7)
    # Row 1: raw heatmap
    _plot_col(1, col, data, cmap, overlay=False, title=title, label_size=7)
    # Row 2: overlay (repeat — mirrors the atlas showing all views)
    _plot_col(2, col, data, cmap, overlay=True,  title=None)

# ── Row 3: colorbars
for ci, (_, cmap, _, cblabel) in enumerate(biomarkers):
    col = ci + 1
    ax_cb = fig1.add_subplot(gs[3, col])
    ax_cb.set_facecolor(DARK_BG)
    norm = Normalize(vmin=0, vmax=1)
    sm   = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = plt.colorbar(sm, cax=ax_cb, orientation="horizontal")
    cb.ax.tick_params(colors="white", labelsize=6)
    cb.ax.set_xlabel(cblabel, color="#aaaaaa", fontsize=6, fontfamily="monospace")
    cb.outline.set_edgecolor("#444444")

# blank colorbar cell under RGB
ax_cb0 = fig1.add_subplot(gs[3, 0])
ax_cb0.set_facecolor(DARK_BG)
ax_cb0.axis("off")

fig1.suptitle("Full Spectral Biomarker Atlas — Foot Tissue",
              color="white", fontsize=13, fontfamily="monospace", y=0.97)

# Bottom status bar
status_txt = (
    f"StO\u2082: {mean_StO2:.3f}  |  Perf(norm): {mean_perf:.3f}  |  "
    f"Mean Risk: {mean_risk:.3f}  |  Zone Imbal: {zone_imbal:.3f}  |  "
    f"Assessment: {assessment}"
)
fig1.text(0.5, 0.005, status_txt, ha="center", va="bottom",
          color=acolor_mpl, fontsize=8, fontfamily="monospace",
          fontweight="bold")

fig1.savefig(os.path.join(OUTDIR, "fig1_atlas.png"), dpi=150, bbox_inches="tight",
             facecolor=DARK_BG)
print("Saved fig1_atlas.png")

# ═════════════════════════════════════════════
# FIGURE 2 — Zone-wise Foot Analysis
# ═════════════════════════════════════════════

fig2 = plt.figure(figsize=(22, 9), facecolor=DARK_BG)
gs2  = GridSpec(1, 4, figure=fig2,
                wspace=0.06, left=0.02, right=0.98, top=0.88, bottom=0.08)

ZONE_COLORS = ["#3cb371", "#d4e88a", "#cc3300"]   # toe=green mid=yellow heel=red
ZONE_NAMES  = ["TOE", "MID", "HEEL"]

# ── Panel 0: Zone legend strip
ax_legend = fig2.add_subplot(gs2[0, 0])
ax_legend.set_facecolor(DARK_BG)
ax_legend.axis("off")
ax_legend.set_title("Foot Zones", color="white", fontsize=10,
                    fontfamily="monospace", pad=6)

zone_heights = [1/3, 1/3, 1/3]
zone_y       = [2/3, 1/3, 0]
for i, (name, ystart, col) in enumerate(zip(ZONE_NAMES, zone_y, ZONE_COLORS)):
    rect = mpatches.FancyBboxPatch(
        (0.10, ystart + 0.01), 0.80, 1/3 - 0.02,
        boxstyle="round,pad=0.01",
        linewidth=1.5, edgecolor="white", facecolor=col,
        transform=ax_legend.transAxes, clip_on=False
    )
    ax_legend.add_patch(rect)
    ax_legend.text(0.50, ystart + 1/6, name,
                   ha="center", va="center",
                   color="white", fontsize=14, fontweight="bold",
                   fontfamily="monospace",
                   transform=ax_legend.transAxes)

# Dashed zone boundary lines on legend
for yline in [1/3, 2/3]:
    ax_legend.axhline(y=yline, color="white", linestyle="--", lw=1, alpha=0.6)

# ── Helper: draw image panel with zone lines + labels
def zone_panel(ax, img_data, cmap=None, is_rgb=False,
               title="", add_zone_text=True, show_values=None):
    ax.set_facecolor(DARK_BG)
    ax.axis("off")
    ax.set_title(title, color="white", fontsize=10,
                 fontfamily="monospace", pad=6)

    H = img_data.shape[0]

    if is_rgb:
        ax.imshow(img_data)
        # Overlay zone color borders
        zone_cols_rgba = ["#00FFFF", "#00FFFF"]
        for frac in [1/3, 2/3]:
            y_px = int(H * frac)
            ax.axhline(y=y_px, color="#00FFFF", linestyle="--", lw=1.5, alpha=0.9)
        # Zone text labels on RGB
        for zn, yfrac, col in zip(ZONE_NAMES,
                                  [1/6, 1/2, 5/6],
                                  ["cyan", "cyan", "cyan"]):
            ax.text(0.5, 1 - yfrac, zn,
                    transform=ax.transAxes,
                    ha="center", va="center",
                    color="cyan", fontsize=12, fontweight="bold",
                    fontfamily="monospace")
    else:
        im = ax.imshow(img_data, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        # Colorbar on right
        divider_ax = ax.inset_axes([1.02, 0, 0.06, 1])
        divider_ax.set_facecolor(DARK_BG)
        norm = Normalize(0, 1)
        sm   = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cb = plt.colorbar(sm, cax=divider_ax)
        cb.ax.tick_params(colors="white", labelsize=7)
        cb.outline.set_edgecolor("#555555")
        for spine in divider_ax.spines.values():
            spine.set_visible(False)

    # Zone dividers
    for frac in [1/3, 2/3]:
        y_px = int(H * frac) if not is_rgb else int(H * frac)
        ax.axhline(y=y_px, color="white", linestyle="--", lw=1.0, alpha=0.55)

    # Value labels
    if show_values:
        for (label, val), yfrac in zip(show_values, [1/6, 1/2, 5/6]):
            ax.text(0.5, 1 - yfrac + 0.05, label,
                    transform=ax.transAxes,
                    ha="center", va="center",
                    color="white", fontsize=8, fontfamily="monospace",
                    fontweight="bold",
                    bbox=dict(facecolor="none", edgecolor="none"))
            ax.text(0.5, 1 - yfrac - 0.03, f"{val:.3f}",
                    transform=ax.transAxes,
                    ha="center", va="center",
                    color="white", fontsize=9, fontfamily="monospace",
                    fontweight="bold")


# ── Panel 1: RGB + zones overlay (with cyan zone color borders)
ax_rgb = fig2.add_subplot(gs2[0, 1])
zone_panel(ax_rgb, rgb, is_rgb=True, title="RGB + Zones")

# ── Panel 2: StO2 map with zone values
ax_sto2 = fig2.add_subplot(gs2[0, 2])
zone_panel(ax_sto2, StO2_n, cmap=plt.cm.RdYlGn, title="StO\u2082 by Zone",
           show_values=[("StO\u2082", toe_sto2_mean),
                        ("StO\u2082", mid_sto2_mean),
                        ("StO\u2082", heel_sto2_mean)])

# ── Panel 3: Risk map with zone values
ax_risk = fig2.add_subplot(gs2[0, 3])
zone_panel(ax_risk, Risk_n, cmap=plt.cm.RdYlGn_r, title="Ulcer Risk by Zone",
           show_values=[("Risk", toe_risk_mean),
                        ("Risk", mid_risk_mean),
                        ("Risk", heel_risk_mean)])

fig2.suptitle("Zone-wise Foot Analysis — Toe / Mid / Heel",
              color="white", fontsize=13, fontfamily="monospace", y=0.96)

status2 = (
    f"Toe \u2192 StO\u2082: {toe_sto2_mean:.3f}  Risk: {toe_risk_mean:.3f}  |  "
    f"Mid \u2192 StO\u2082: {mid_sto2_mean:.3f}  Risk: {mid_risk_mean:.3f}  |  "
    f"Heel \u2192 StO\u2082: {heel_sto2_mean:.3f}  Risk: {heel_risk_mean:.3f}  |  "
    f"Zone Imbalance: {zone_imbal:.3f}  |  Assessment: {assessment}"
)
fig2.text(0.5, 0.01, status2, ha="center", va="bottom",
          color=acolor_mpl, fontsize=8, fontfamily="monospace", fontweight="bold")

fig2.savefig(os.path.join(OUTDIR, "fig2_zones.png"), dpi=150, bbox_inches="tight",
             facecolor=DARK_BG)
print("Saved fig2_zones.png")

# ═════════════════════════════════════════════
# FIGURE 3 — Clinical Summary (dark terminal style)
# ═════════════════════════════════════════════

fig3, ax3 = plt.subplots(figsize=(13, 8), facecolor=DARK_BG)
ax3.set_facecolor(DARK_BG)
ax3.axis("off")

MONO  = "monospace"
WHITE = "#d8d8d8"
CYAN  = "#00e5ff"
GOLD  = acolor_mpl

cx = 0.06   # left margin in axes fraction
lh = 0.078  # line height

def hline(ax, y, x0=0.03, x1=0.58, color="#555555", lw=0.8):
    ax.plot([x0, x1], [y, y], color=color, lw=lw,
            transform=ax.transAxes, clip_on=False)

def tline(ax, y, text, color=WHITE, size=12, bold=False, x=None):
    ax.text(x if x else cx, y, text,
            transform=ax.transAxes,
            color=color, fontsize=size, fontfamily=MONO,
            fontweight="bold" if bold else "normal",
            va="top")

y = 0.95
tline(ax3, y, "SPECTRAL FOOT ANALYSIS — CLINICAL SUMMARY", WHITE, 14, bold=True)
y -= 0.06
hline(ax3, y + 0.01)
y -= 0.06

tline(ax3, y, "Overall Assessment :  " + assessment, GOLD, 13, bold=True)
y -= 0.05
hline(ax3, y + 0.01)
y -= 0.06
 
y -= 0.05
hline(ax3, y + 0.01)
y -= 0.05

metrics = [
    ("Mean StO\u2082 (proxy)",    f"{mean_StO2:.4f}"),
    ("Mean Perfusion (norm)",     f"{mean_perf:.4f}"),
    ("Mean Risk Score",           f"{mean_risk:.4f}"),
    ("Max  Risk Score",           f"{max_risk:.4f}"),
    (f"Zone Imbalance (StO\u2082)",f"{zone_imbal:.4f}"),
]

for label, val in metrics:
    ax3.text(cx, y, label, transform=ax3.transAxes,
             color=WHITE, fontsize=11, fontfamily=MONO, va="top")
    ax3.text(cx + 0.32, y, ":  " + val, transform=ax3.transAxes,
             color=WHITE, fontsize=11, fontfamily=MONO, va="top")
    y -= lh

y -= 0.02
hline(ax3, y + 0.01)
y -= 0.04

zone_rows = [
    (f"Toe  \u2192 StO\u2082: {toe_sto2_mean:.3f}",  f"Risk: {toe_risk_mean:.3f}"),
    (f"Mid  \u2192 StO\u2082: {mid_sto2_mean:.3f}",  f"Risk: {mid_risk_mean:.3f}"),
    (f"Heel \u2192 StO\u2082: {heel_sto2_mean:.3f}", f"Risk: {heel_risk_mean:.3f}"),
]

for z_label, z_risk in zone_rows:
    ax3.text(cx, y, z_label, transform=ax3.transAxes,
             color=CYAN, fontsize=11, fontfamily=MONO, va="top", fontweight="bold")
    ax3.text(cx + 0.28, y, z_risk, transform=ax3.transAxes,
             color=CYAN, fontsize=11, fontfamily=MONO, va="top", fontweight="bold")
    y -= lh

fig3.savefig(os.path.join(OUTDIR, "fig3_clinical.png"), dpi=150, bbox_inches="tight",
             facecolor=DARK_BG)
print("Saved fig3_clinical.png")

# ═════════════════════════════════════════════
# plt.show() for interactive use (comment out if running headless)
# ═════════════════════════════════════════════
plt.show()

# ═════════════════════════════════════════════
# PDF REPORT
# ═════════════════════════════════════════════

PDF_PATH = os.path.join(OUTDIR, "clinical_foot_report.pdf")

def add_image_page(c, img_path, page_w, page_h, title="", margin=20):
    """Add a full-page image with optional title header."""
    c.setFillColorRGB(0.05, 0.05, 0.05)
    c.rect(0, 0, page_w, page_h, fill=1, stroke=0)

    if title:
        c.setFont("Courier-Bold", 13)
        c.setFillColorRGB(0.85, 0.85, 0.85)
        c.drawCentredString(page_w / 2, page_h - 26, title)

    top_offset = 36 if title else margin
    avail_w = page_w - 2 * margin
    avail_h = page_h - top_offset - margin

    reader = ImageReader(img_path)
    iw, ih = reader.getSize()
    scale  = min(avail_w / iw, avail_h / ih)
    draw_w = iw * scale
    draw_h = ih * scale
    x0 = (page_w - draw_w) / 2
    y0 = margin + (avail_h - draw_h) / 2

    c.drawImage(reader, x0, y0, draw_w, draw_h)


# A3 landscape gives more breathing room
from reportlab.lib.pagesizes import A3
PAGE_W, PAGE_H = landscape(A3)

c = rl_canvas.Canvas(PDF_PATH, pagesize=(PAGE_W, PAGE_H))

# ── Cover page ────────────────────────────────
c.setFillColorRGB(0.05, 0.05, 0.05)
c.rect(0, 0, PAGE_W, PAGE_H, fill=1, stroke=0)

c.setFont("Courier-Bold", 28)
c.setFillColorRGB(0.85, 0.85, 0.85)
c.drawCentredString(PAGE_W / 2, PAGE_H / 2 + 80,
                    "SPECTRAL FOOT ANALYSIS")
c.setFont("Courier-Bold", 18)
c.setFillColorRGB(0.60, 0.60, 0.60)
c.drawCentredString(PAGE_W / 2, PAGE_H / 2 + 38,
                    "Clinical Report")

# Assessment line in color
acolor_r = {"lime": (0, 1, 0), "yellow": (1, 0.84, 0),
             "orange": (1, 0.65, 0), "red": (1, 0, 0)}[acolor_mpl]
c.setFillColorRGB(*acolor_r)
c.setFont("Courier-Bold", 15)
c.drawCentredString(PAGE_W / 2, PAGE_H / 2 - 10, assessment)

c.setFont("Courier", 11)
c.setFillColorRGB(0.5, 0.5, 0.5)
date_str = datetime.datetime.now().strftime("%d %B %Y  %H:%M")
c.drawCentredString(PAGE_W / 2, PAGE_H / 2 - 50, f"Generated: {date_str}")

c.showPage()

# ── Page 2: Clinical Summary ──────────────────
add_image_page(c, os.path.join(OUTDIR, "fig3_clinical.png"), PAGE_W, PAGE_H,
               title="Clinical Summary")
c.showPage()

# ── Page 3: Zone Analysis ─────────────────────
add_image_page(c, os.path.join(OUTDIR, "fig2_zones.png"), PAGE_W, PAGE_H,
               title="Zone-wise Foot Analysis")
c.showPage()

c.save()
print(f"\nPDF report saved")
print("Done.")

Bands loaded successfully
RGB shape: (2321, 1224, 3)
Saved fig1_atlas.png
